# Testing de aplicación web Flask con PostgreSQL

### En este notebook,  se muestra cómo se reorganizaron las vistas de la aplicación web, separando responsabilidades y preparando el proyecto para ser escalable y mantenible. Asimismo se presenta la generación de tests para una aplicación web

## 📁 Estructura del proyecto

```
08_proy_appweb/
├── appweb/
│   ├── __init__.py
│   ├── models.py
│   ├── postgres_db.py
│   ├── sistema.py
│   ├── static/
│   │   └── styles.css
│   ├── templates/
│   │   ├── consulta.html
│   │   ├── inicio.html
│   │   └── login.html
│   └── views.py
├── tests/
│   ├── __init__.py
│   └── test_sistema.py
└── pytest.ini
```

---

## Archivo `postgres_db.py` - Conexión a PostgreSQL con pool de conexiones

### Contiene el manejo a la conexión a la base de datos usando `psycopg2` con un pool de conexiones. Incluye la creación de tablas y un administrador de contexto para cursores.

In [ ]:

from psycopg2.pool import ThreadedConnectionPool, PoolError
from contextlib import contextmanager
import atexit


db_config = { "host" : "localhost",
                "database" : "sistema_abc",
                "user" : "uacm",
                "password" : "uacm1"}


_tabla_productos = "CREATE TABLE Productos (" \
                "id SERIAL PRIMARY KEY,"  \
                "descripcion TEXT,"   \
                "precio NUMERIC(10, 2)" \
                ");"


class PostgresDB:
    def __init__(self):
        self.app = None
        self.pool = None
        self._closed = False 

    def init_app(self, app):
        self.app = app
        self.connect()
        atexit.register(self.close)                

    def connect(self):
        self.pool = ThreadedConnectionPool(minconn=1, maxconn=30, **db_config)

    def create_all_tables(self):
        drop_productos ="DROP TABLE IF EXISTS Productos;"
        with self.get_cursor() as cur: 
            cur.execute(drop_productos)
            cur.execute(_tabla_productos)


# El decorador @contextmanager, se utiliza para crear administradores de contexto (context managers), 
# lo que facilita la gestión de recursos de manera más limpia y eficiente. 
# Se puede definir el comportamiento de entrada y salida del contexto dentro de una función, en lugar de tener que definir una clase con los métodos __enter__ y __exit__
    @contextmanager
    def get_cursor(self):
        if self.pool is None:
            self.connect()
        # Obtiene una conexión del pool
        con = self.pool.getconn()
        cur = con.cursor()
        try:
            # "pausa" y entrega el cursor al bloque with
            yield cur
            # Al salir del with
            # Se ejecuta si no hubo excepciones
            con.commit()
        except Exception:
            con.rollback()  # cancelar los cambios
            raise
        finally:
            cur.close()          # cerrar el cursor
            self.pool.putconn(con)  # devolver conexión al pool

    def close(self):
        if self.pool and not self._closed:
            try:
                self.pool.closeall()
            except PoolError as e:
                # Si ya está cerrado, ignoramos
                if "connection pool is closed" not in str(e):
                    raise
            finally:
                self._closed = True
                self.pool = None

# Instancia a la DB

pgdb = PostgresDB()            

## Archivo `models.py` - Consultas a la base de datos

### Contiene las funciones que interactúan directamente con la base de datos, como la consulta de productos.

In [ ]:
# models.py
from appweb.postgres_db import pgdb

# Consultar los productos de la base de datos
def consulta_prod():
    resultados = []
    try:
        # Obtener una conexión
        with pgdb.get_cursor() as cursor:
            # Ejecutar una consulta
            cursor.execute("SELECT * FROM productos")
            # Obtener los resultados
            filas = cursor.fetchall()
            # Mostrar los resultados
            for fila in filas:
                print(fila)
                resultados.append(fila)
    except Exception as e:
        print("Error:", e)

    return resultados


## Archivo `views.py` - Definición de rutas y lógica de vistas

### Contiene la definición de las rutas de la aplicación y se conectan con los modelos y plantillas.

In [ ]:
# views.py
from flask import render_template, request, redirect, url_for
from appweb.models import consulta_prod

# Datos de usuarios (simulados para el ejemplo)
USUARIOS = {"usuario1": "u1", "usuario2": "u2"}


def registrar_rutas(app):
    # ----------------------------------------
    # Página de registro al sistema
    # ----------------------------------------
    @app.route("/", methods=["GET", "POST"])
    @app.route("/login", methods=["GET", "POST"])
    def login_home():
        error = None
        if request.method == "POST":
            username = request.form.get("usuario")
            password = request.form.get("contrasena")
            if username in USUARIOS and USUARIOS[username] == password:
                # Iniciar sesión exitosa, redirigir a otra página
                return redirect(url_for("inicio_home"))
            else:
                # Credenciales incorrectas, mostrar mensaje de error
                # return 'Credenciales incorrectas. <a href="/login">Intenta de nuevo</a>'
                msj = "Credenciales incorrectas"
                return render_template("login.html", error=msj), 401
        else:
            return render_template("login.html")

    # ----------------------------------------
    # Página  del menu principal
    # ----------------------------------------
    @app.route("/inicio", methods=["GET", "POST"])
    def inicio_home():
        return render_template("inicio.html")

    # ----------------------------------------
    # Página de la consulta de productos
    # ----------------------------------------
    @app.route("/consulta_productos", methods=["GET", "POST"])
    def consulta_productos():
        resultados = consulta_prod()
        return render_template("consulta.html", datos=resultados)


## Archivo `sistema.py` - Fábrica de la aplicación Flask

### Contiene la creación de la instancia principal de Flask y registra las rutas (endpoints).

In [ ]:
# sistema.py
from flask import Flask
from appweb.postgres_db import pgdb
from appweb.views import registrar_rutas

def create_app():
    app = Flask(__name__)
    registrar_rutas(app)
    return app

app = create_app()
pgdb.init_app(app)

if __name__ == "__main__":
    app.run(debug=True)

## Archivo  `templates/login.html` - Página de inicio de sesión

In [ ]:
<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Iniciar sesión</title>
    
    <link rel="stylesheet" href="https://maxcdn.bootstrapcdn.com/bootstrap/3.4.1/css/bootstrap.min.css">

</head>
<body style="background-color: #c4bf1d;">

    <div class="container"> 

    <h1>Iniciar sesión</h1>
    {% if error %}
    <p><strong>Error:</strong> {{ error }}
    {% endif %}
    <form action="/login" method="post">
        <div class="form-group">
            <label for="username" style="display: block; font-weight: bold;">Usuario:</label>
            <input type="text" id="usuario" name="usuario"><br>
            <label for="password" style="display: block; font-weight: bold;">Contraseña:</label>
            <input type="password" id="contrasena" name="contrasena"><br>
        </div>

        
        <button type="submit" >Iniciar sesión</button>


    </form>
</div>    
</body>
</html>

## Archivo : `templates/inicio.html` - Menú principal

### Contiene el menú principal de la aplicación web

In [ ]:

<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Inicio</title>
    
</head>
<body>
    <h1>Menú</h1>
    <table>
        <tr>
            <p style="color:blue;"> <a href="{{url_for('consulta_productos')}}">Consultar productos</a> </p>        
        </tr>
    </table>
</body>
</html>


## Archivo `templates/consulta.html` - Listado de productos

### Contienen la lógica que despliega al usuario las consultas realizadas a la base de datos

In [ ]:
<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Consulta</title>
    
</head>
<body>
    <h1>Lista de Datos</h1>
    <table>
        <tr>
            <th>ID</th>
            <th>DESCRIPCÓN</th>
            <th>PRECIO</th>
        </tr>
        {% for id, desc, precio in datos %}
        <tr>
            <td>{{ id }}</td>
            <td>{{ desc }}</td>
            <td>{{ precio }}</td>
        </tr>
        {% endfor %}
    </table>
</body>
</html>


## Archivo `static/styles.css` - Estilos personalizados
### Contiene los estilos que se usan en las plantillas html

In [ ]:
.body {
    background-color: #f4f404; /* Gris claro */
}

.container {
    background-color: #41089d; /* Blanco */
}

.form-group label {
    color: #aa1515; /* Gris oscuro */
}

.btn {
    background-color: #007bff; /* Azul */
    color: #21ba3f; /* Blanco */
}

.btn:hover {
    background-color: #0056b3; /* Azul oscuro */
}

## Archivo `__init__.py` - Para que `appweb` sea un paquete Python

In [ ]:
# Este archivo convierte a la carpeta 'appweb' en un paquete Python

## Archivo de Pruebas con `pytest`

### Archivo `tests/test_sistema.py` contiene pruebas básicas de la aplicación.

In [ ]:
import pytest
from appweb.sistema import create_app
from appweb.postgres_db import pgdb

# carga los metodos de las vistas


@pytest.fixture(scope="session")
def app_flask():
    # Setup de la aplicación
    app = create_app()

    app.config.update(
        {
            "TESTING": True,
        }
    )

    # Inicialización de la base de datos
    print("...inicializando entorno de TESTING...")
    pgdb.init_app(app)
    # pgdb.create_all_tables()

    yield app
    # apartir de aquí se pone el código para liberar los recursos

    # teardown limpiar/ reinicializar


@pytest.fixture(scope="session")
def client(app_flask):
    print("creando cliente...")
    with app_flask.test_client() as client:
        # Establecer el contexto de aplicación
        with app_flask.app_context():
            yield client


# --------------------------------------------------
# Test de las transacciones de la aplicación
# --------------------------------------------------


def test_login(client):
    usuario = "usuario1"
    password = "u1"
    response = client.post(
        "/login",
        data={"usuario": usuario, "contrasena": password},
        follow_redirects=True,
    )
    data = response.get_data().decode("utf-8")
    print(data)
    assert "<h1>Menú</h1>" in data
    # assert response.status_code == 200


def test_login_failed(client):

    # Si las credenciales fallaron la página retorna un codigo 401
    usuario = "usuario1"
    password = "u"
    response = client.post(
        "/login",
        data={"usuario": usuario, "contrasena": password},
        follow_redirects=True,
    )
    data = response.get_data().decode("utf-8")
    print(data)
    assert "<strong>Error:</strong> Credenciales incorrectas" in data
    # assert response.status_code == 401


def test_consulta_producto(client):
    response = client.post("/consulta_productos", data={}, follow_redirects=True)
    data = response.get_data().decode("utf-8")
    print(data)
    assert "Lista de Datos" in data


# ===========================================================================
# Ejecución de los test.
# ===========================================================================
# Desde la terminal,  ejecutar en la carpeta "tests"
#
#
# pytest -v test_sistema.py
#
# ---------------------------------------------------------------------------
# Para visualizar las impresiones en consola usar el parámetro -s
# ---------------------------------------------------------------------------
#
# pytest -vs test_sistema.py
#
# ---------------------------------------------------------------------------


# ===========================================================================
# Ejercicio
# ===========================================================================
# De las páginas creadas para la tabla Empleado agregar los tests
# correspondientes.
#
# 1. Agregar los tests para consulta de empleados
# 2. Agregar los tests para agregar empleado
# ===========================================================================


## Archivo `pytest.ini` - Configuración de pytest

### Contiene las rutas 

In [ ]:
[pytest]
pythonpath = appweb
testpaths = tests

## Resumen de la reestructuración

| Archivo | Responsabilidad |
|---------|----------------|
| `sistema.py` | Fábrica de la app Flask |
| `views.py` | Rutas y lógica de vistas |
| `models.py` | Consultas a la base de datos |
| `postgres_db.py` | Conexión y pool de PostgreSQL |
| `templates/` | Plantillas HTML |
| `static/` | Archivos CSS estáticos |
| `tests/` | Pruebas automatizadas |

### Con esta separación permite mantener el código ordenado, facilita las pruebas y hace que la aplicación sea más mantenible.

